In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive')

!mkdir -p olist-customer-intelligence/{data/raw,data/processed,notebooks,sql,src,models,reports,app/pages,docs}
os.chdir('/content/drive/MyDrive/M5-forecasting')
!touch src/__init__.py
print("Now in:", os.getcwd())

Now in: /content/drive/MyDrive/M5-forecasting


In [3]:
%%writefile src/models/lightgbm_model.py
"""LightGBM quantile regression for M5 item-level forecasting."""
import numpy as np
import pandas as pd
import lightgbm as lgb


# Columns that must never be used as features
DROP_COLS = {
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "d", "date", "sales", "wm_yr_wk", "weekday", "year",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
}

CATEGORICAL = ["dow", "month", "snap", "is_weekend", "is_event", "is_christmas"]


def get_feature_cols(df):
    return [c for c in df.columns if c not in DROP_COLS]


def train_quantile(X, y, alpha, categorical, params=None):
    base = dict(
        objective="quantile", alpha=alpha, metric="quantile",
        n_estimators=400, learning_rate=0.05,
        num_leaves=127, min_child_samples=100,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        n_jobs=-1, verbosity=-1,
    )
    if params:
        base.update(params)
    model = lgb.LGBMRegressor(**base)
    model.fit(X, y, categorical_feature=categorical)
    return model


def pinball_loss(y_true, y_pred, alpha):
    diff = y_true - y_pred
    return np.mean(np.maximum(alpha * diff, (alpha - 1) * diff))

Overwriting src/models/lightgbm_model.py


In [4]:
import pandas as pd, numpy as np, pickle
import lightgbm as lgb
from src.models.lightgbm_model import (
    get_feature_cols, train_quantile, pinball_loss, CATEGORICAL, DROP_COLS
)

In [5]:
import pandas as pd, numpy as np, gc
import lightgbm as lgb
from src.models.lightgbm_model import get_feature_cols, pinball_loss, CATEGORICAL, DROP_COLS

# peek at schema to get feature cols without loading all rows
schema = pd.read_parquet("data/processed/features.parquet", columns=["sales"])
del schema; gc.collect()

feat = pd.read_parquet("data/processed/features.parquet")
feat_cols = get_feature_cols(feat)
cats = [c for c in CATEGORICAL if c in feat_cols]
print(feat.shape, "|", len(feat_cols), "features")

(21403980, 47) | 31 features


In [6]:
cutoff = feat["date"].max() - pd.Timedelta(days=28)

# save test to disk first (small)
test = feat.loc[feat["date"] > cutoff, feat_cols + ["sales", "date", "id"]]
test.to_parquet("data/processed/_lgb_test.parquet", index=False)
del test; gc.collect()

# training rows only, 50% subsample chosen up front
train_mask = (feat["date"] <= cutoff).values
sub = np.random.RandomState(42).rand(len(feat)) < 0.5
keep_rows = train_mask & sub

y_train = feat.loc[keep_rows, "sales"].values.astype("float32")
for c in cats:
    feat[c] = feat[c].astype("category")
X_train = feat.loc[keep_rows, feat_cols]

del feat; gc.collect()
print("train rows after subsample:", X_train.shape[0])

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cats, free_raw_data=True)
del X_train; gc.collect()

train rows after subsample: 10273166


0

In [7]:
def q_params(alpha):
    return dict(objective="quantile", alpha=alpha, metric="quantile",
                learning_rate=0.05, num_leaves=127, min_child_samples=100,
                bagging_fraction=0.8, bagging_freq=1, feature_fraction=0.8,
                num_threads=-1, verbosity=-1)

models = {}
for alpha, name in [(0.1,"p10"),(0.5,"p50"),(0.9,"p90")]:
    print(f"training {name}...")
    models[name] = lgb.train(q_params(alpha), dtrain, num_boost_round=400)
    print("  done:", name)

training p10...
  done: p10
training p50...
  done: p50
training p90...
  done: p90


In [8]:
test = pd.read_parquet("data/processed/_lgb_test.parquet")
X_test = test[feat_cols].copy()
for c in cats:
    X_test[c] = X_test[c].astype("category")

pred = pd.DataFrame({
    "id": test["id"].values,
    "date": test["date"].values,
    "sales": test["sales"].values.astype("float32"),
})
for name in ["p10", "p50", "p90"]:
    pred[name] = models[name].predict(X_test)

# clip negatives + enforce monotonicity (no quantile crossing)
q = pred[["p10","p50","p90"]].clip(lower=0).values
q.sort(axis=1)
pred[["p10","p50","p90"]] = q
print(pred.head())

                            id       date  sales  p10       p50       p90
0  FOODS_1_001_CA_1_evaluation 2016-04-25    2.0  0.0  0.996431  2.814275
1  FOODS_1_001_CA_1_evaluation 2016-04-26    0.0  0.0  0.996243  2.530950
2  FOODS_1_001_CA_1_evaluation 2016-04-27    0.0  0.0  0.992066  2.485720
3  FOODS_1_001_CA_1_evaluation 2016-04-28    0.0  0.0  0.704538  2.434704
4  FOODS_1_001_CA_1_evaluation 2016-04-29    0.0  0.0  0.705552  2.712722


In [9]:
for name, alpha in [("p10",0.1),("p50",0.5),("p90",0.9)]:
    pl = pinball_loss(pred["sales"].values, pred[name].values, alpha)
    print(f"pinball {name}: {pl:.4f}")

p50_rmse = np.sqrt(((pred["p50"] - pred["sales"])**2).mean())
print("P50 RMSE (item-level):", round(p50_rmse, 3))

cov = ((pred["sales"] >= pred["p10"]) & (pred["sales"] <= pred["p90"])).mean()
print("P10-P90 coverage:", round(cov, 3), "(target 0.80)")

pinball p10: 0.1316
pinball p50: 0.4543
pinball p90: 0.2913
P50 RMSE (item-level): 2.05
P10-P90 coverage: 0.881 (target 0.80)


In [10]:
imp = pd.Series(models["p50"].feature_importance(), index=feat_cols).sort_values(ascending=False)
print(imp.head(15))

roll_mean_56         10052
roll_mean_7           6476
roll_zero_frac_56     5248
roll_mean_14          3870
wday                  3623
lag_1                 3487
roll_mean_28          3432
roll_zero_frac_28     2843
is_christmas          1979
roll_std_56           1602
roll_std_14           1462
roll_std_7            1456
dow_sin               1312
roll_std_28           1036
sell_price             481
dtype: int32


In [15]:
import pickle
pred.to_parquet("data/processed/forecasts_lightgbm.parquet", index=False)
for name in ["p10","p50","p90"]:
    with open(f"models/lightgbm_quantile_{name}.pkl","wb") as f:
        pickle.dump(models[name], f)
print("saved")

saved


## LightGBM Quantile Regression

**Models:** 3 LightGBM boosters, quantile objective (α = 0.1 / 0.5 / 0.9), item level (all 30,490 series). Native `lgb.train` API for memory efficiency.

**Setup:**
- Time-series split: last 28 days held out, no lookahead
- 50% row subsample on training (Colab memory) — test set untouched, evaluation honest
- Predictions clipped ≥ 0, row-sorted to prevent quantile crossing

**Results (28-day item-level test):**
- Pinball — P10: 0.132, P50: 0.454, P90: 0.291
- P50 RMSE: 2.05 (item-level; NOT comparable to Prophet's aggregate 209.8)
- **P10–P90 coverage: 0.881** vs 0.80 target → mildly over-covered (intervals slightly conservative; addressed in calibration)

**Feature importance — validates the EDA:**
- Top: roll_mean_56, roll_mean_7 (demand level, two timescales)
- **roll_zero_frac_56 / _28 rank 3rd & 8th** → intermittency features (from 69.4% finding) carry real signal
- is_christmas rank 9 → closure modeling matters

**Saved:** `forecasts_lightgbm.parquet`, 3× `lightgbm_quantile_{p10,p50,p90}.pkl`

**Log to limitations.md:** 50% row subsample for compute; full-data training is a straightforward scale-up.

In [1]:
!nvidia-smi

Thu Jul  2 21:52:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----